## Strategy Diff Report

This notebook evaluates how far the simulation values are from the targets established in the NDC document

In [1]:
import pandas as pd
import os

In [2]:
SCRIPT_DIR_PATH = os.getcwd()
PARENT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
TABLEAU_DIR_PATH = os.path.join(PARENT_DIR_PATH, "tableau/data")
DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")

In [3]:
tableau_decomposed_df = pd.read_csv(os.path.join(TABLEAU_DIR_PATH, "decomposed_emissions_bulgaria_2022.csv"))
tableau_decomposed_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.101736,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.101736,0.101736
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.101529,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.101529,0.101529
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.101320,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.101320,0.101320
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.101108,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.101108,0.101108
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.100891,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.100891,0.100891


In [4]:
# Check ippu just to be sure data is correct
tableau_decomposed_df[tableau_decomposed_df["CSC.Sector"] == 'Energy']

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
232,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.415001,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.415001,NaN
233,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.414772,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.414772,NaN
234,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.415659,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.415659,NaN
235,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.416381,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.416381,NaN
236,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.417646,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.417646,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4544,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.076789,2018,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.076789,NaN
4545,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.080076,2019,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.080076,NaN
4546,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.076704,2020,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.076704,NaN
4547,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.081371,2021,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.081371,NaN


In [5]:
# Check Energy for WEM
tableau_decomposed_df[(tableau_decomposed_df["CSC.Sector"] == 'Energy') & (tableau_decomposed_df["strategy"] == 'WAM') & (tableau_decomposed_df["Year"] == 2050)].to_clipboard()

In [6]:
# Aggregate by CSC.Sector and Year fields and sum value
agg_df = tableau_decomposed_df.groupby(['CSC.Sector', 'Year', 'strategy'])['value'].sum().reset_index()
agg_df = agg_df.rename(columns={"CSC.Sector": "Subsector"})

# Make column name lowercase
agg_df.columns = agg_df.columns.str.lower()
agg_df.head()

,subsector,year,strategy,value
0,Agriculture,2000,Historical,5.415444
1,Agriculture,2001,Historical,5.150073
2,Agriculture,2002,Historical,4.694411
3,Agriculture,2003,Historical,4.824542
4,Agriculture,2004,Historical,5.040389


In [7]:
agg_df.subsector.unique()

array(['Agriculture', 'CCSQ', 'Energy', 'Industrial Processes',
       'Land Use, Land Use Change, and Forestry', 'Waste'], dtype=object)

In [8]:
# Filter out years outside relevant years
relevant_years = [2022, 2050]
agg_df = agg_df[agg_df['year'].isin(relevant_years)]
agg_df.head()

,subsector,year,strategy,value
22,Agriculture,2022,Historical,6.666645
23,Agriculture,2022,Strategy TX:BASE,6.666645
24,Agriculture,2022,WAM,6.666645
25,Agriculture,2022,WEM,6.666645
107,Agriculture,2050,Strategy TX:BASE,7.521852


In [9]:
# Filter out CCSQ
agg_df = agg_df[~agg_df['subsector'].str.contains("CCSQ")]
agg_df.head()

,subsector,year,strategy,value
22,Agriculture,2022,Historical,6.666645
23,Agriculture,2022,Strategy TX:BASE,6.666645
24,Agriculture,2022,WAM,6.666645
25,Agriculture,2022,WEM,6.666645
107,Agriculture,2050,Strategy TX:BASE,7.521852


In [10]:
# Filter out Historical strategy
agg_df = agg_df[~agg_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
agg_df.head()

,subsector,year,strategy,value
24,Agriculture,2022,WAM,6.666645
25,Agriculture,2022,WEM,6.666645
108,Agriculture,2050,WAM,4.955091
109,Agriculture,2050,WEM,5.627323
221,Energy,2022,WAM,46.437527


In [11]:
# Sort it by strategy and year
agg_df = agg_df.sort_values(by=['strategy', 'year'], ascending=[False, True])
agg_df.head()

,subsector,year,strategy,value
25,Agriculture,2022,WEM,6.666645
222,Energy,2022,WEM,46.437527
332,Industrial Processes,2022,WEM,8.847664
442,"Land Use, Land Use Change, and Forestry",2022,WEM,-9.887958
552,Waste,2022,WEM,6.393792


In [12]:
wide = (
    agg_df
    .pivot(
        index="subsector",
        columns=["year", "strategy"],
        values="value"
    )
)

# flatten the MultiIndex columns → "2022_LEP", "2022_WAM", etc.
wide.columns = [f"{year}_{strategy}" for year, strategy in wide.columns]

wide = wide.reset_index()

In [13]:
wide

,subsector,2022_WEM,2050_WEM,2022_WAM,2050_WAM
0,Agriculture,6.666645,5.627323,6.666645,4.955091
1,Energy,46.437527,15.641751,46.437527,4.275158
2,Industrial Processes,8.847664,4.544166,8.847664,1.670632
3,"Land Use, Land Use Change, and Forestry",-9.887958,-11.409579,-9.887958,-11.421666
4,Waste,6.393792,2.686233,6.393792,1.311573


In [14]:
# Drop 2022_WAM column as not relevant
wide = wide.drop(columns=["2022_WAM"])

In [15]:
# Load the report template
template_df = pd.read_csv(os.path.join(DATA_DIR_PATH, "strategy_report_template.csv"))
template_df

,subsector,2022_target,2030_WEM_target,2050_WEM_target,2030_WAM_target,2050_WAM_target
0,Energy,45.1,23.8,15.1,16.7,0.0
1,"Land Use, Land Use Change, and Forestry",-9.3,-9.5,-9.2,-9.5,-9.2
2,Waste,2.8,2.1,1.9,1.6,1.4
3,Agriculture,5.5,5.7,6.0,5.5,5.8
4,Industrial Processes,4.6,4.0,3.9,3.4,0.5


In [16]:
# Merge with the template
report_df = template_df.merge(wide, how='left', on='subsector')
report_df

,subsector,2022_target,2030_WEM_target,2050_WEM_target,2030_WAM_target,2050_WAM_target,2022_WEM,2050_WEM,2050_WAM
0,Energy,45.1,23.8,15.1,16.7,0.0,46.437527,15.641751,4.275158
1,"Land Use, Land Use Change, and Forestry",-9.3,-9.5,-9.2,-9.5,-9.2,-9.887958,-11.409579,-11.421666
2,Waste,2.8,2.1,1.9,1.6,1.4,6.393792,2.686233,1.311573
3,Agriculture,5.5,5.7,6.0,5.5,5.8,6.666645,5.627323,4.955091
4,Industrial Processes,4.6,4.0,3.9,3.4,0.5,8.847664,4.544166,1.670632


In [17]:
# Add a total row
total_row = pd.DataFrame(report_df.select_dtypes(include='number').sum()).T
total_row['subsector'] = 'Total'
report_df = pd.concat([report_df, total_row], ignore_index=True)
report_df

,subsector,2022_target,2030_WEM_target,2050_WEM_target,2030_WAM_target,2050_WAM_target,2022_WEM,2050_WEM,2050_WAM
0,Energy,45.1,23.8,15.1,16.7,0.0,46.437527,15.641751,4.275158
1,"Land Use, Land Use Change, and Forestry",-9.3,-9.5,-9.2,-9.5,-9.2,-9.887958,-11.409579,-11.421666
2,Waste,2.8,2.1,1.9,1.6,1.4,6.393792,2.686233,1.311573
3,Agriculture,5.5,5.7,6.0,5.5,5.8,6.666645,5.627323,4.955091
4,Industrial Processes,4.6,4.0,3.9,3.4,0.5,8.847664,4.544166,1.670632
5,Total,48.7,26.1,17.7,17.7,-1.5,58.457669,17.089893,0.790788


In [18]:
# Absolute errors instead of relative errors
report_df["ae_2022"] = (report_df["2022_target"] - report_df["2022_WEM"]).abs()

report_df["ae_2050_WEM"] = (report_df["2050_WEM_target"] - report_df["2050_WEM"]).abs()

report_df["ae_2050_WAM"] = (report_df["2050_WAM_target"] - report_df["2050_WAM"]).abs()


In [19]:
report_df

,subsector,2022_target,2030_WEM_target,2050_WEM_target,2030_WAM_target,2050_WAM_target,2022_WEM,2050_WEM,2050_WAM,ae_2022,ae_2050_WEM,ae_2050_WAM
0,Energy,45.1,23.8,15.1,16.7,0.0,46.437527,15.641751,4.275158,1.337527,0.541751,4.275158
1,"Land Use, Land Use Change, and Forestry",-9.3,-9.5,-9.2,-9.5,-9.2,-9.887958,-11.409579,-11.421666,0.587958,2.209579,2.221666
2,Waste,2.8,2.1,1.9,1.6,1.4,6.393792,2.686233,1.311573,3.593792,0.786233,0.088427
3,Agriculture,5.5,5.7,6.0,5.5,5.8,6.666645,5.627323,4.955091,1.166645,0.372677,0.844909
4,Industrial Processes,4.6,4.0,3.9,3.4,0.5,8.847664,4.544166,1.670632,4.247664,0.644166,1.170632
5,Total,48.7,26.1,17.7,17.7,-1.5,58.457669,17.089893,0.790788,9.757669,0.610107,2.290788


In [20]:
report_df.to_csv(os.path.join(TABLEAU_DIR_PATH, "strategy_diff_report.csv"), index=False)